# Final Project SQL Queries

This notebook contains the required SQL analysis for the final project. The goal is to use SQL to support the main analysis by summarizing umbrella favorability, joining weather data with Google Trends data, ranking time periods, and using window functions to study changes over time.

Each query includes comments explaining what it does, how it works, and why it is relevant.

In [1]:
import pandas as pd
import sqlite3

conn = sqlite3.connect("umbrella_final_project.db")

weather_hourly = pd.read_csv("weather_hourly.csv")
weather_daily = pd.read_csv("weather_daily.csv")
weather_weekly = pd.read_csv("weather_weekly.csv")
google_trends_weekly = pd.read_csv("google_trends_weekly.csv")
weekly_combined = pd.read_csv("weekly_combined.csv")
monthly_combined = pd.read_csv("monthly_combined.csv")

weather_hourly.to_sql("weather_hourly", conn, if_exists="replace", index=False)
weather_daily.to_sql("weather_daily", conn, if_exists="replace", index=False)
weather_weekly.to_sql("weather_weekly", conn, if_exists="replace", index=False)
google_trends_weekly.to_sql("google_trends_weekly", conn, if_exists="replace", index=False)
weekly_combined.to_sql("weekly_combined", conn, if_exists="replace", index=False)
monthly_combined.to_sql("monthly_combined", conn, if_exists="replace", index=False)

37

In [2]:
def run_query(query):
    return pd.read_sql_query(query, conn)

In [3]:
query_1 = """
-- Query 1: Check size and basic validity of the hourly weather dataset.
-- This query counts total rows and checks whether any core weather fields are missing.
-- This matters because the umbrella favorability framework depends on temperature,
-- precipitation, and wind speed being available.

SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN temperature_2m IS NULL THEN 1 ELSE 0 END) AS missing_temperature,
    SUM(CASE WHEN precipitation IS NULL THEN 1 ELSE 0 END) AS missing_precipitation,
    SUM(CASE WHEN wind_speed_10m IS NULL THEN 1 ELSE 0 END) AS missing_wind_speed
FROM weather_hourly;
"""

run_query(query_1)

,total_rows,missing_temperature,missing_precipitation,missing_wind_speed
0,26304,0,0,0


In [4]:
query_2 = """
-- Query 2: Calculate average daily umbrella favorability by month.
-- This uses GROUP BY to identify which months are most umbrella-friendly.
-- This supports the time-series and seasonality portion of the analysis.

SELECT
    month,
    ROUND(AVG(umbrella_value_score), 2) AS avg_daily_favorability,
    ROUND(AVG(precipitation), 2) AS avg_daily_precipitation
FROM weather_daily
GROUP BY month
ORDER BY month;
"""

run_query(query_2)

,month,avg_daily_favorability,avg_daily_precipitation
0,1,-0.88,0.09
1,2,-0.54,0.03
2,3,1.06,0.13
3,4,-0.37,0.11
4,5,1.57,0.09
5,6,1.73,0.11
6,7,3.10,0.25
7,8,1.53,0.10
8,9,2.36,0.12
9,10,-0.09,0.10


In [5]:
query_3 = """
-- Query 3: Calculate average daily umbrella favorability by season.
-- This uses GROUP BY to compare broader seasonal patterns.
-- This helps test whether umbrella usefulness is better understood seasonally.

SELECT
    season,
    ROUND(AVG(umbrella_value_score), 2) AS avg_daily_favorability,
    ROUND(AVG(precipitation), 2) AS avg_daily_precipitation
FROM weather_daily
GROUP BY season
ORDER BY avg_daily_favorability DESC;
"""

run_query(query_3)

,season,avg_daily_favorability,avg_daily_precipitation
0,Summer,2.12,0.15
1,Spring,0.77,0.11
2,Fall,0.77,0.10
3,Winter,-0.31,0.07


In [6]:
query_4 = """
-- Query 4: Join weekly weather data with weekly Google Trends data.
-- This is an INNER JOIN on week_start.
-- It connects objective umbrella favorability with public search interest.

SELECT
    w.week_start,
    w.umbrella_value_score,
    w.precipitation,
    w.temperature_2m,
    w.wind_speed_10m,
    g.umbrella_search_interest
FROM weather_weekly AS w
INNER JOIN google_trends_weekly AS g
    ON w.week_start = g.week_start
ORDER BY w.week_start;
"""

run_query(query_4)

,week_start,umbrella_value_score,precipitation,temperature_2m,wind_speed_10m,umbrella_search_interest
0,2023-03-26,-5,0.052,40.216667,16.112500,20
1,2023-04-02,-7,1.507,46.466667,12.539881,24
2,2023-04-09,0,0.032,61.539286,10.782143,35
3,2023-04-16,-25,0.704,47.992262,14.217262,26
4,2023-04-23,7,0.400,43.563095,8.724405,28
...,...,...,...,...,...,...
153,2026-03-01,24,0.710,39.595238,6.811905,25
154,2026-03-08,14,0.403,43.107738,10.483929,24
155,2026-03-15,-21,1.972,38.282738,9.157738,25
156,2026-03-22,-9,0.297,41.509524,9.341667,27


In [7]:
query_5 = """
-- Query 5: Identify the weeks with the highest umbrella search interest.
-- This query joins Google Trends to weather data so that each search spike
-- can be interpreted alongside actual weather conditions.

SELECT
    g.week_start,
    g.umbrella_search_interest,
    w.umbrella_value_score,
    w.precipitation,
    w.temperature_2m,
    w.wind_speed_10m
FROM google_trends_weekly AS g
INNER JOIN weather_weekly AS w
    ON g.week_start = w.week_start
ORDER BY g.umbrella_search_interest DESC
LIMIT 10;
"""

run_query(query_5)

,week_start,umbrella_search_interest,umbrella_value_score,precipitation,temperature_2m,wind_speed_10m
0,2024-08-11,100,25,1.871,72.355357,8.255357
1,2024-08-04,71,-4,0.213,71.988095,10.092857
2,2024-08-18,56,0,0.032,67.465476,8.748214
3,2023-05-28,51,4,0.099,68.681548,7.737500
4,2023-05-21,45,0,0.000,59.173214,10.113095
5,2024-06-16,45,-2,0.198,80.821429,12.848810
6,2023-05-07,42,42,1.281,56.523810,8.426190
7,2023-06-18,42,0,0.000,69.905952,9.064286
8,2025-06-22,42,25,0.873,80.360714,7.282143
9,2023-07-02,41,16,7.315,69.981548,7.463095


In [8]:
query_6 = """
-- Query 6: Identify the weeks with the highest weather-based umbrella favorability.
-- This query joins weather data to Google Trends data to see whether the best
-- umbrella weeks also had high public search interest.

SELECT
    w.week_start,
    w.umbrella_value_score,
    g.umbrella_search_interest,
    w.precipitation,
    w.temperature_2m,
    w.wind_speed_10m
FROM weather_weekly AS w
INNER JOIN google_trends_weekly AS g
    ON w.week_start = g.week_start
ORDER BY w.umbrella_value_score DESC
LIMIT 10;
"""

run_query(query_6)

,week_start,umbrella_value_score,umbrella_search_interest,precipitation,temperature_2m,wind_speed_10m
0,2025-06-15,49,37,2.091,73.779762,6.950000
1,2023-05-07,42,42,1.281,56.523810,8.426190
2,2024-09-22,41,27,1.987,65.036905,11.570238
3,2023-09-10,38,21,2.197,62.884524,6.844048
4,2025-07-06,38,33,1.486,74.548810,5.979762
5,2024-01-21,36,16,1.401,30.500000,8.437500
6,2025-07-20,36,31,0.657,76.764881,6.314881
7,2023-06-25,35,36,2.707,72.156548,9.422024
8,2024-07-07,34,36,1.788,72.900000,7.286905
9,2025-08-17,33,26,0.691,72.934524,6.499405


In [9]:
query_7 = """
-- Query 7: Find weeks where umbrella favorability was above the overall weekly average.
-- This uses a subquery to calculate the average weekly favorability score.
-- It helps identify weeks that were unusually favorable for umbrella use.

SELECT
    week_start,
    umbrella_value_score,
    precipitation,
    temperature_2m,
    wind_speed_10m
FROM weather_weekly
WHERE umbrella_value_score > (
    SELECT AVG(umbrella_value_score)
    FROM weather_weekly
)
ORDER BY umbrella_value_score DESC;
"""

run_query(query_7)

,week_start,umbrella_value_score,precipitation,temperature_2m,wind_speed_10m
0,2025-06-15,49,2.091,73.779762,6.950000
1,2023-05-07,42,1.281,56.523810,8.426190
2,2024-09-22,41,1.987,65.036905,11.570238
3,2023-09-10,38,2.197,62.884524,6.844048
4,2025-07-06,38,1.486,74.548810,5.979762
...,...,...,...,...,...
65,2024-11-17,7,1.078,46.110119,9.713690
66,2023-10-22,6,1.058,57.355952,12.429167
67,2025-08-24,6,0.156,65.163690,6.008333
68,2025-12-21,6,0.112,37.798810,6.980952


In [10]:
query_8 = """
-- Query 8: Find weeks where Google search interest was above average.
-- This uses a subquery to calculate the average search interest.
-- It helps identify periods when public attention toward umbrellas was unusually high.

SELECT
    week_start,
    umbrella_search_interest
FROM google_trends_weekly
WHERE umbrella_search_interest > (
    SELECT AVG(umbrella_search_interest)
    FROM google_trends_weekly
)
ORDER BY umbrella_search_interest DESC;
"""

run_query(query_8)

,week_start,umbrella_search_interest
0,2024-08-11,100
1,2024-08-04,71
2,2024-08-18,56
3,2023-05-28,51
4,2023-05-21,45
...,...,...
60,2024-03-31,26
61,2024-04-07,26
62,2025-08-17,26
63,2026-03-01,25


In [11]:
query_9 = """
-- Query 9: Calculate a 4-week rolling average of umbrella favorability.
-- This uses a SQL window function to smooth short-term volatility.
-- It supports the finding that broader time horizons reveal clearer patterns.

SELECT
    week_start,
    umbrella_value_score,
    ROUND(
        AVG(umbrella_value_score) OVER (
            ORDER BY week_start
            ROWS BETWEEN 3 PRECEDING AND CURRENT ROW
        ), 2
    ) AS rolling_4_week_favorability
FROM weather_weekly
ORDER BY week_start;
"""

run_query(query_9)

,week_start,umbrella_value_score,rolling_4_week_favorability
0,2023-03-26,-5,-5.00
1,2023-04-02,-7,-6.00
2,2023-04-09,0,-4.00
3,2023-04-16,-25,-9.25
4,2023-04-23,7,-6.25
...,...,...,...
153,2026-03-01,24,4.50
154,2026-03-08,14,8.00
155,2026-03-15,-21,3.25
156,2026-03-22,-9,2.00


In [12]:
query_10 = """
-- Query 10: Rank months by average umbrella favorability.
-- This uses a window function with RANK().
-- It identifies the most umbrella-friendly months in the dataset.

SELECT
    year,
    month,
    ROUND(AVG(umbrella_value_score), 2) AS avg_monthly_favorability,
    RANK() OVER (
        ORDER BY AVG(umbrella_value_score) DESC
    ) AS favorability_rank
FROM weather_daily
GROUP BY year, month
ORDER BY favorability_rank;
"""

run_query(query_10)

,year,month,avg_monthly_favorability,favorability_rank
0,2025,7,4.26,1
1,2025,6,3.87,2
2,2023,9,3.10,3
3,2023,7,2.52,4
4,2024,7,2.52,4
5,2025,9,2.50,6
6,2025,3,2.29,7
7,2025,5,2.06,8
8,2025,8,2.06,8
9,2024,6,1.53,10
